# AnIML Eval — golden_compare.py against the target-grounded induction

Scores the induced ontology from the COA AnIML run against the **AnIML/AFO golden ontology**, reusing the repo's own `coa_ontology.benchmark.golden_compare` (per-class / per-property precision / recall / F1).

**Run context**
- Env: `agentops-publisher` (195698601974), us-east-1
- Namespace: `animl-afo` = `3a6c6a53-af55-43fe-bcb9-8ea7ebee55e9`
- Proposal job: `25416fff-d3d5-4f65-b101-6cfbf3d6f9cb`
- Strategy `table_to_ontology`, `grounding_mode=ENHANCED`, grounding ontology `http://www.w3id.org/animl/ontology/`

**⚠️ READ THIS before trusting the numbers.** `golden_compare.py` matches entities by **local-name + fuzzy string**, and for *properties* it does **NOT** traverse `owl:equivalentProperty`. Our induced property local-names are `measurements_value` etc.; the match to `aml:pointValue` lives in an `owl:equivalentProperty` **edge** the scorer ignores for scoring. So **property recall/F1 will read LOW even though the mapping is correct**. The real signal (34 column equivalences) is counted directly in the last cell. This gap is itself a finding for the COA eng sync: to score a grounding result, the scorer must follow the equivalence edges — or score `matches.json`/`ontology.ttl` equivalences directly.

**How to run:** kernel = the repo venv (`.venv/bin/python`). Requires AWS creds for the `agentops-publisher` profile for the S3 pull cell (or drop a local `ontology.ttl` beside this notebook and skip it).

In [ ]:
# Cell 1 — paths & config
import json
import subprocess
from pathlib import Path

REPO = Path.cwd()
while REPO.parent != REPO and not (REPO / "packages" / "ontology-engine").exists():
    REPO = REPO.parent
assert (REPO / "packages" / "ontology-engine").exists(), f"run from inside the repo; got {Path.cwd()}"

PROFILE = "agentops-publisher"
REGION = "us-east-1"
BUCKET = "coa-dev-ontology-artifacts-195698601974"
NS = "3a6c6a53-af55-43fe-bcb9-8ea7ebee55e9"
JOB = "25416fff-d3d5-4f65-b101-6cfbf3d6f9cb"

WORK = REPO / "specs" / "danaher" / "prep"
INDUCED = WORK / "eval_artifacts" / "induced_ontology.ttl"
MATCHES = WORK / "eval_artifacts" / "matches.json"
GOLDEN = WORK / "animl_data" / "animl_ontology.owl"  # rdflib parses .owl/.rdf as RDF/XML
INDUCED.parent.mkdir(parents=True, exist_ok=True)
print("repo:", REPO)
print("golden exists:", GOLDEN.exists())

In [ ]:
# Cell 2 — pull the induced ontology + matches.json from S3 (skip if you already have local copies)
def s3cp(key, dest):
    """Copy one proposal artifact from S3 to a local path."""
    src = f"s3://{BUCKET}/proposals/{NS}/{JOB}/latest/{key}"
    r = subprocess.run(
        ["aws", "s3", "cp", src, str(dest), "--profile", PROFILE, "--region", REGION], capture_output=True, text=True
    )
    print(("OK " if r.returncode == 0 else "FAIL ") + src, "->", dest.name)
    if r.returncode:
        print(r.stderr.strip()[:300])


if not INDUCED.exists():
    s3cp("ontology.ttl", INDUCED)
if not MATCHES.exists():
    s3cp("matches.json", MATCHES)
print("induced:", INDUCED.exists(), INDUCED.stat().st_size if INDUCED.exists() else 0, "bytes")

In [ ]:
# Cell 3 — load both graphs
import sys

sys.path.insert(0, str(REPO / "packages" / "ontology-engine" / "src"))
from coa_ontology.benchmark import compare_to_golden
from rdflib import Graph

induced_g = Graph().parse(str(INDUCED), format="turtle")
gfmt = "xml" if GOLDEN.suffix in (".owl", ".rdf") else "turtle"
golden_g = Graph().parse(str(GOLDEN), format=gfmt)
print(f"induced triples: {len(induced_g)}   golden triples: {len(golden_g)}")

In [ ]:
# Cell 4 — score with the repo's golden_compare
score = compare_to_golden(induced_g, golden_g)

print(json.dumps(score.as_dict(), indent=2))

The scorecard above is the **name-overlap** view. Expect **class** numbers to look reasonable (classes carry `owl:equivalentClass`/`subClassOf` edges the scorer DOES traverse for the `grounded_to` field) and **property** numbers to look **understated** — that is the caveat, not a failure. The next cell counts the real result the scorer cannot see.

In [ ]:
# Cell 5 — the real result: owl:equivalentProperty edges the scorer does NOT traverse
from rdflib import OWL

eq = [(str(s), str(o)) for s, o in induced_g.subject_objects(OWL.equivalentProperty)]
print(f"owl:equivalentProperty axioms: {len(eq)}  (pre-fix run had 0)")


def ln(u):
    """Return the local name of an IRI."""
    for sep in ("#", "/"):
        if sep in u:
            u = u.rsplit(sep, 1)[-1]
    return u


for s, o in sorted(eq):
    print(f"  {ln(s):40s} -> aml:{ln(o)}")

In [ ]:
# Cell 6 — table groundings from matches.json (owl:equivalentClass view)
m = json.loads(MATCHES.read_text())
for r in m.get("matches", []):
    print(
        f"{r.get('source_table'):18s} -> {r.get('matched_class_uri')}  "
        f"sim={r.get('similarity')} {r.get('relationship')} [{r.get('scoring_strategy')}]"
    )

## Takeaways
1. **golden_compare.py is reusable** as the coverage/precision scorecard — reuse it, don't build a parallel harness.
2. **But it undercounts a grounding result**: property matching is name/fuzzy only and ignores `owl:equivalentProperty`. For the Danaher eval, either extend it to follow equivalence edges, or score `matches.json` + the equivalence axioms directly (Cell 5).
3. **The actual proof** = Cell 5's 34 column equivalences + Cell 6's 3 table groundings, plus the live governed NL query (304 rows, Tier 2 ontology) recorded in `RESULT-animl-run.md` §0.
4. **Not scored here** — semantic correctness (SME-judged) and transformation expressions (NOT COA; the harness/Mapper layer).